# 05 — Load Balancing: Examples

> Distribute requests across multiple keys and providers to avoid rate limits.

---

**What you'll build:**
1. Round-robin balancer
2. Weighted balancer
3. Least-latency balancer
4. Rate-limit-aware balancer
5. Full load-balanced router with fallback

In [ ]:
# !pip install openai python-dotenv rich

In [ ]:
import os, time, statistics
from collections import deque, defaultdict
from itertools import cycle
from dataclasses import dataclass, field
from openai import OpenAI
from dotenv import load_dotenv
from rich.table import Table
from rich.console import Console
from rich import print as rprint

load_dotenv()
client = OpenAI()
console = Console()
print('✅ Setup complete')

---
## Part 1: Round-Robin Balancer

In [ ]:
class RoundRobinBalancer:
    """Cycles through endpoints in equal turns."""

    def __init__(self, endpoints: list[str]):
        self.endpoints = endpoints
        self._cycle = cycle(endpoints)
        self._counts: dict[str, int] = defaultdict(int)

    def next(self) -> str:
        ep = next(self._cycle)
        self._counts[ep] += 1
        return ep

    def distribution(self) -> dict:
        total = sum(self._counts.values())
        return {ep: f"{count}/{total} ({count/total*100:.0f}%)" for ep, count in self._counts.items()}


# ── Simulation ────────────────────────────────────────────────────────────
rr = RoundRobinBalancer(["key_A", "key_B", "key_C"])

assignments = [rr.next() for _ in range(12)]
print("Round-robin assignments:", assignments)
print("\nDistribution:")
for ep, dist in rr.distribution().items():
    print(f"   {ep}: {dist}")

---
## Part 2: Weighted Balancer

In [ ]:
class WeightedBalancer:
    """
    Endpoints receive traffic proportional to their weight.
    weights=[3, 2, 1] → 50%, 33%, 17% of traffic.
    """

    def __init__(self, endpoints: list[str], weights: list[int]):
        assert len(endpoints) == len(weights), "Lengths must match"
        pool = []
        for ep, w in zip(endpoints, weights):
            pool.extend([ep] * w)
        self._cycle = cycle(pool)
        self._counts: dict[str, int] = defaultdict(int)
        self.endpoints = endpoints
        self.weights = weights

    def next(self) -> str:
        ep = next(self._cycle)
        self._counts[ep] += 1
        return ep

    def distribution(self) -> dict:
        total = sum(self._counts.values()) or 1
        return {ep: f"{self._counts[ep]}/{total} ({self._counts[ep]/total*100:.0f}%)" for ep in self.endpoints}


# ── Simulation ────────────────────────────────────────────────────────────
wb = WeightedBalancer(
    endpoints=["tier1_key", "tier2_key", "free_key"],
    weights=[4, 2, 1]   # Tier1 has most quota
)

for _ in range(70): wb.next()
print("Weighted distribution (70 requests):")
for ep, dist in wb.distribution().items():
    print(f"   {ep}: {dist}")

---
## Part 3: Least-Latency Balancer

In [ ]:
class LeastLatencyBalancer:
    """Routes to the endpoint with the lowest rolling average latency."""

    def __init__(self, endpoints: list[str], window: int = 10):
        self.endpoints = endpoints
        self._latencies: dict[str, deque] = {ep: deque(maxlen=window) for ep in endpoints}

    def record(self, endpoint: str, latency_ms: float):
        """Record a latency observation for an endpoint."""
        self._latencies[endpoint].append(latency_ms)

    def avg_latency(self, endpoint: str) -> float:
        hist = self._latencies[endpoint]
        return statistics.mean(hist) if hist else float("inf")

    def next(self) -> str:
        """Return endpoint with lowest average latency."""
        return min(self.endpoints, key=self.avg_latency)

    def stats(self) -> None:
        table = Table(title="Endpoint Latency Stats", show_header=True)
        table.add_column("Endpoint", style="cyan")
        table.add_column("Avg Latency", style="yellow")
        table.add_column("Samples", style="white")
        table.add_column("Selected Next", style="green")
        best = self.next()
        for ep in self.endpoints:
            lat = self.avg_latency(ep)
            table.add_row(
                ep,
                f"{lat:.0f}ms" if lat != float('inf') else "(no data)",
                str(len(self._latencies[ep])),
                "✅" if ep == best else ""
            )
        console.print(table)


# ── Simulation ────────────────────────────────────────────────────────────
import random
random.seed(42)

llb = LeastLatencyBalancer(["provider_A", "provider_B", "provider_C"])

# Simulate latency observations
latency_profiles = {"provider_A": (800, 50), "provider_B": (1200, 100), "provider_C": (600, 30)}
for _ in range(10):
    for ep, (mean, std) in latency_profiles.items():
        llb.record(ep, max(100, random.gauss(mean, std)))

llb.stats()

---
## Part 4: Rate-Limit-Aware Balancer

In [ ]:
class RateLimitAwareBalancer:
    """Routes to the endpoint with the most remaining capacity."""

    def __init__(self, endpoints: list[str], rpm_limits: dict[str, int]):
        self.endpoints = endpoints
        self.rpm_limits = rpm_limits
        self._requests: dict[str, list[float]] = defaultdict(list)

    def _current_rpm(self, endpoint: str) -> int:
        """Count requests made in the last 60 seconds."""
        now = time.time()
        self._requests[endpoint] = [t for t in self._requests[endpoint] if t > now - 60]
        return len(self._requests[endpoint])

    def capacity(self, endpoint: str) -> int:
        limit = self.rpm_limits.get(endpoint, 60)
        return max(0, limit - self._current_rpm(endpoint))

    def next(self) -> str | None:
        """Return endpoint with most remaining capacity, or None if all saturated."""
        capacities = {ep: self.capacity(ep) for ep in self.endpoints}
        best = max(capacities, key=capacities.get)
        if capacities[best] == 0:
            return None  # All saturated
        self._requests[best].append(time.time())
        return best

    def status(self):
        table = Table(title="Endpoint Capacity (Rate-Limit Aware)", show_header=True)
        table.add_column("Endpoint", style="cyan")
        table.add_column("RPM Limit", style="white")
        table.add_column("Used", style="yellow")
        table.add_column("Remaining", style="green")
        for ep in self.endpoints:
            used = self._current_rpm(ep)
            cap = self.capacity(ep)
            limit = self.rpm_limits.get(ep, 60)
            bar = "█" * int(used / limit * 10) + "░" * (10 - int(used / limit * 10))
            table.add_row(ep, str(limit), f"{used} {bar}", str(cap))
        console.print(table)


# ── Simulation ────────────────────────────────────────────────────────────
rlb = RateLimitAwareBalancer(
    endpoints=["key_A", "key_B", "key_C"],
    rpm_limits={"key_A": 100, "key_B": 60, "key_C": 30}
)

# Simulate 80 requests
selected = []
for _ in range(80):
    ep = rlb.next()
    selected.append(ep)

rlb.status()

from collections import Counter
print("\nSelected counts:", dict(Counter(selected)))

---
## Part 5: Full Load-Balanced Router

In [ ]:
@dataclass
class LoadBalancedResponse:
    content: str
    model: str
    endpoint: str
    latency_ms: float
    fallback_used: bool = False


class LoadBalancedRouter:
    """
    Combines:
    - Rate-limit-aware balancing across multiple endpoints
    - Latency tracking
    - Fallback to alternative model if all primary endpoints are saturated
    """

    def __init__(
        self,
        primary_model: str,
        api_keys: list[str],
        rpm_limits: dict[str, int] | None = None,
        fallback_model: str = "gpt-4o-mini",
    ):
        self.primary_model = primary_model
        self.fallback_model = fallback_model
        self.api_keys = api_keys

        # Use key IDs as endpoint labels
        endpoints = [f"key_{i}" for i in range(len(api_keys))]
        default_limits = {ep: 60 for ep in endpoints}
        self.balancer = RateLimitAwareBalancer(endpoints, rpm_limits or default_limits)
        self.latency_tracker = LeastLatencyBalancer(endpoints)
        self._key_map = dict(zip(endpoints, api_keys))

    def _get_client(self, endpoint: str) -> OpenAI:
        return OpenAI(api_key=self._key_map[endpoint])

    def complete(self, messages: list[dict], model: str | None = None) -> LoadBalancedResponse:
        target_model = model or self.primary_model

        # Try load-balanced endpoint
        endpoint = self.balancer.next()
        if endpoint:
            try:
                api_client = self._get_client(endpoint)
                start = time.time()
                resp = api_client.chat.completions.create(
                    model=target_model, messages=messages, max_tokens=150
                )
                latency = (time.time() - start) * 1000
                self.latency_tracker.record(endpoint, latency)
                return LoadBalancedResponse(
                    content=resp.choices[0].message.content,
                    model=target_model, endpoint=endpoint,
                    latency_ms=round(latency, 1)
                )
            except Exception as e:
                rprint(f"   ❌ [{endpoint}] Failed: {e}")

        # Fallback: use default client + fallback model
        rprint(f"   ⚡ All primary keys saturated — using fallback model: {self.fallback_model}")
        start = time.time()
        resp = client.chat.completions.create(
            model=self.fallback_model, messages=messages, max_tokens=150
        )
        latency = (time.time() - start) * 1000
        return LoadBalancedResponse(
            content=resp.choices[0].message.content,
            model=self.fallback_model, endpoint="fallback",
            latency_ms=round(latency, 1), fallback_used=True
        )


# ── Demo using single key (same key repeated for demo) ────────────────────
api_key = os.getenv("OPENAI_API_KEY", "")

lb_router = LoadBalancedRouter(
    primary_model="gpt-4o-mini",
    api_keys=[api_key, api_key],  # In production: different keys
    rpm_limits={"key_0": 500, "key_1": 500},
    fallback_model="gpt-4o-mini"
)

prompts = [
    "What is load balancing?",
    "Explain rate limiting in APIs.",
    "What is exponential backoff?",
]

print("\n📤 Running load-balanced requests...\n")
for prompt in prompts:
    result = lb_router.complete([{"role": "user", "content": prompt}])
    rprint(f"   [cyan]{prompt}[/cyan]")
    rprint(f"   → endpoint={result.endpoint}, model={result.model}, latency={result.latency_ms}ms")
    rprint(f"   → {result.content[:80]}...\n")

---
## Summary

| Strategy | Best For |
|---|---|
| Round-robin | Even, predictable traffic |
| Weighted | Keys with different quota sizes |
| Least-latency | Real-time / low-latency requirements |
| Rate-limit-aware | High volume, avoiding 429 errors |
| Combined router | Production: health + capacity + fallback |

**Next:** `06_litellm_router` — production-grade routing built into LiteLLM.